# 08 - Periods and regimes

Step 8 of `THESIS_GUIDE.md` (PIANO.md C3, reduced). The rolling protocol gives five test years with different markets (chapter 3, Market Regimes): **2020** (the COVID-19 crash) and **2022** (the rate-driven bear market) are the high-volatility years of every universe, 2021, 2023 and 2024 the calm ones. This notebook asks whether the models predict, invest and *explain* differently in the two regimes:

1. predictive skill (RankIC) per test year and regime, against the market's volatility;
2. the portfolios' return over the equal-weight benchmark per year;
3. the families the models rely on (IG, net family shares) per year and regime;
4. how much the explanations drift from one year to the next.

Core grid, T = 20, L = 5, every seed available; sections 3-4 read the explanations saved by `scripts/explain.py`.

In [1]:
import json
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

from xaifin import viz
from xaifin.analysis.variance import load_importance, pair_agreement
from xaifin.config import BENCHMARK_INDEX, CORE_UNIVERSES, TOP_K, UNIVERSE_NAMES
from xaifin.data.features import market_names
from xaifin.data.summary import summarize, volatility_by_year
from xaifin.portfolio.backtest import find_portfolios
from xaifin.training.rolling import collect_results

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
SEQ_LEN, PRED_LEN = 20, 5
MODELS = viz.MODEL_ORDER
REGIME = {2020: "high volatility", 2021: "calm", 2022: "high volatility", 2023: "calm", 2024: "calm"}
runs = collect_results()
runs = runs[(runs["seq_len"] == SEQ_LEN) & (runs["pred_len"] == PRED_LEN) & runs["universe"].isin(CORE_UNIVERSES)]
runs["regime"] = runs["test_year"].map(REGIME)
volatility = pd.concat({u: volatility_by_year(summarize(u)["market_volatility"])[BENCHMARK_INDEX[u]] for u in CORE_UNIVERSES},
                       names=["universe", "test_year"]).rename("volatility").reset_index()
print(f"{len(runs)} runs; index volatility per test year:")
volatility.pivot(index="test_year", columns="universe", values="volatility").loc[2020:2024].style.format("{:.1%}")

252 runs; index volatility per test year:


universe,dji,nasdaq100,sx5e
test_year,,,
2020,28.3%,30.5%,27.1%
2021,12.0%,17.2%,14.3%
2022,19.2%,32.1%,22.0%
2023,11.4%,18.3%,13.9%
2024,10.9%,17.4%,12.9%


## 1. Predictive skill per year and regime

Mean test RankIC per model and year (over universes and seeds), the same per regime, and the share of runs with a positive RankIC in each regime. The scatter puts each universe-year's mean RankIC (over models and seeds) against the volatility of its index.

In [2]:
by_year = runs.pivot_table(index="model", columns="test_year", values="RankIC", aggfunc="mean").reindex(MODELS)
display(by_year.style.format("{:+.4f}").background_gradient(cmap="RdBu", vmin=-0.04, vmax=0.04, axis=None))
regime = (runs.groupby(["model", "regime"])["RankIC"]
          .agg(mean="mean", positive=lambda s: (s > 0).mean(), runs="size").unstack("regime").reindex(MODELS))
display(regime)
cells = runs.groupby(["universe", "test_year"])["RankIC"].mean().reset_index().merge(volatility, on=["universe", "test_year"])
cells["universe"] = cells["universe"].map(UNIVERSE_NAMES)
fig = px.scatter(cells, x="volatility", y="RankIC", color="universe", text="test_year",
                 color_discrete_sequence=viz.THEMES[False]["series"], title="Mean RankIC of a universe-year against the volatility of its index")
fig.update_traces(textposition="top center")
fig.update_xaxes(tickformat=".0%")
fig.show()

test_year,2020,2021,2022,2023,2024
model,,,,,
MASTER,+0.0285,+0.0103,-0.0124,+0.0044,+0.0003
MATCC,+0.0316,+0.0049,-0.0086,+0.0131,-0.0038
FactorVAE,+0.0226,+0.0206,-0.0434,+0.0103,-0.0449
HIST,+0.0350,+0.0045,-0.0012,+0.0149,+0.0141
DiscoverPLF,+0.0114,+0.0012,+0.0012,+0.0218,-0.0036
FinFormer,+0.0384,+0.0190,-0.0030,+0.0060,-0.0036


mean                 positive                 runs  \
regime         calm high volatility     calm high volatility calm   
model                                                               
MASTER       0.0050          0.0039   0.5926          0.4667   27   
MATCC        0.0047          0.0075   0.5926          0.6000   27   
FactorVAE   -0.0046         -0.0170   0.4444          0.3333   27   
HIST         0.0111          0.0133   0.5926          0.6667   27   
DiscoverPLF  0.0065          0.0052   0.7037          0.8000   27   
FinFormer    0.0071          0.0136   0.6296          0.7333   27   

                             
regime      high volatility  
model                        
MASTER                   15  
MATCC                    15  
FactorVAE                15  
HIST                     15  
DiscoverPLF              15  
FinFormer                15

## 2. Portfolio returns per year

Return of each top-k portfolio minus the equal-weight benchmark of the same days, per calendar year, mean over universes and seeds.

In [3]:
portfolios = find_portfolios()
portfolios = portfolios[(portfolios["seq_len"] == SEQ_LEN) & (portfolios["pred_len"] == PRED_LEN)
                        & portfolios["universe"].isin(CORE_UNIVERSES) & (portfolios["k"] == portfolios["universe"].map(TOP_K))]
rows = []
for r in portfolios.itertuples():
    yearly = json.loads((r.path / "metrics.json").read_text())["yearly_returns"]
    for year, value in yearly["portfolio"].items():
        rows.append({"model": r.model, "universe": r.universe, "seed": r.seed, "test_year": int(year),
                     "excess": value - yearly["equal_weight"][year]})
excess = pd.DataFrame(rows)
excess["regime"] = excess["test_year"].map(REGIME)
display(excess.pivot_table(index="model", columns="test_year", values="excess", aggfunc="mean").reindex(MODELS).style.format("{:+.1%}"))
def beats_benchmark(s):
    return (s > 0).mean()

excess.pivot_table(index="model", columns="regime", values="excess", aggfunc=["mean", beats_benchmark]).reindex(MODELS)

test_year,2020,2021,2022,2023,2024
model,,,,,
MASTER,+5.3%,-0.9%,+0.9%,+5.6%,-5.6%
MATCC,+4.4%,-2.1%,-3.1%,-1.0%,-8.0%
FactorVAE,+4.2%,-7.9%,-8.5%,+4.2%,-4.7%
HIST,-2.6%,-3.5%,-5.6%,+2.1%,-5.8%
DiscoverPLF,-6.2%,+1.3%,-2.1%,+3.9%,-7.5%
FinFormer,+6.1%,+2.2%,-0.1%,-1.2%,-10.7%


mean                 beats_benchmark                
regime         calm high volatility            calm high volatility
model                                                              
MASTER      -0.0030          0.0265          0.4444          0.6667
MATCC       -0.0368         -0.0009          0.2963          0.5333
FactorVAE   -0.0281         -0.0343          0.4444          0.3333
HIST        -0.0238         -0.0441          0.4074          0.2000
DiscoverPLF -0.0077         -0.0379          0.4444          0.2000
FinFormer   -0.0326          0.0241          0.3704          0.5333

## 3. Which families the models rely on, per year

Net family shares of IG (score target), mean over universes and seeds, per model and test year; then the difference between the high-volatility and the calm years. The market family is left out (zero with the day-mean baseline).

In [4]:
family = load_importance("ig_joint", "pred", "family", "net")
family = family[(family["pred_len"] == PRED_LEN) & family["universe"].isin(CORE_UNIVERSES) & (family["name"] != "market")]
print(f"{family.groupby(['model', 'universe', 'seed', 'test_year']).ngroups} explained runs")
family["regime"] = family["test_year"].map(REGIME)
shares = family.groupby(["model", "test_year", "name"])["importance"].mean().reset_index()
px.line(shares, x="test_year", y="importance", color="name", facet_col="model", facet_col_wrap=3, markers=True,
        category_orders={"model": MODELS}, color_discrete_sequence=viz.THEMES[False]["series"] + ["#898781"],
        title="Net share of the IG attribution per family and test year").show()
shift = family.pivot_table(index=["model", "name"], columns="regime", values="importance", aggfunc="mean")
shift["high - calm"] = shift["high volatility"] - shift["calm"]
shift.reindex(MODELS, level=0)

252 explained runs


regime                     calm  high volatility  high - calm
model       name                                             
MASTER      candle       0.0792           0.0933       0.0141
            momentum     0.1925           0.1624      -0.0301
            position     0.2059           0.2152       0.0093
            price_volume 0.0997           0.0871      -0.0126
            trend        0.1589           0.1744       0.0155
            volatility   0.1247           0.1284       0.0037
            volume       0.1391           0.1392       0.0001
MATCC       candle       0.0779           0.0834       0.0055
            momentum     0.1841           0.1719      -0.0122
            position     0.1922           0.1913      -0.0009
            price_volume 0.1288           0.1218      -0.0070
            trend        0.1454           0.1598       0.0144
            volatility   0.1369           0.1346      -0.0023
            volume       0.1341           0.1364       0.0023
FactorVAE   candle       0.0749           0.0780       0.0031
            momentum     0.2081           0.1792      -0.0289
            position     0.1956           0.1908      -0.0048
            price_volume 0.1189           0.1070      -0.0119
            trend        0.1303           0.1484       0.0181
            volatility   0.1440           0.1604       0.0164
            volume       0.1282           0.1362       0.0080
HIST        candle       0.0054           0.0116       0.0062
            momentum     0.6926           0.7070       0.0144
            volume       0.3020           0.2814      -0.0206
DiscoverPLF candle       0.0448           0.0422      -0.0026
            momentum     0.6783           0.6967       0.0184
            volume       0.2769           0.2610      -0.0159
FinFormer   candle       0.0387           0.0311      -0.0076
            momentum     0.7868           0.7852      -0.0016
            volume       0.1745           0.1836       0.0092

## 4. Year-to-year drift of the explanations

Agreement (Spearman, family and feature level) between the explanations of the same model, universe and seed in two consecutive test years: low agreement means the model relies on different inputs from one year to the next. Transitions into and out of a high-volatility year are compared with calm-to-calm transitions.

In [5]:
market = {(u, n) for u in CORE_UNIVERSES for n in market_names(u)} | {(u, "market") for u in CORE_UNIVERSES}
rows = []
for level, value in (("family", "net"), ("feature", "importance")):
    table = load_importance("ig_joint", "pred", level, value)
    table = table[(table["pred_len"] == PRED_LEN) & table["universe"].isin(CORE_UNIVERSES)]
    table = table[[(u, n) not in market for u, n in zip(table["universe"], table["name"])]]  # zero by construction
    pairs = pair_agreement(table)
    pairs = pairs[pairs["factor"] == "year"].astype({"a": int, "b": int})
    pairs = pairs[(pairs["b"] - pairs["a"]).abs() == 1]
    pairs["transition"] = np.where(pairs["a"].map(REGIME).eq("calm") & pairs["b"].map(REGIME).eq("calm"), "calm to calm",
                                   "involves a high-volatility year")
    rows.append(pairs.assign(level=level))
drift = pd.concat(rows)
display(drift.groupby(["level", "transition"])[["spearman", "kendall"]].mean())
drift.groupby(["level", "model"])["spearman"].mean().unstack("level").reindex(MODELS)

spearman  kendall
level   transition                                        
family  calm to calm                       0.9226   0.8783
        involves a high-volatility year    0.9348   0.8876
feature calm to calm                       0.7060   0.5517
        involves a high-volatility year    0.6959   0.5412

level,family,feature
model,,
MASTER,0.9026,0.7189
MATCC,0.9426,0.8101
FactorVAE,0.7435,0.5630
HIST,1.0000,0.9481
DiscoverPLF,1.0000,0.8121
FinFormer,1.0000,0.3396


## Observations

Core grid, T = 20, L = 5, 252 runs. Regimes follow the volatility of each universe's index: 27–31% in 2020 and 19–32% in 2022, against 11–18% in the calm years. EURO STOXX 50 has no test year 2020, so the 2020 figures average DJI and NASDAQ-100 only.

**1. "High volatility" is not one regime for predictive skill.** 2020 is the best test year for five of the six models: their mean RankIC is +0.023 to +0.038, and only DiscoverPLF does better in 2023. 2022 is negative for five of the six, from −0.001 (HIST) to −0.043 (FactorVAE), with DiscoverPLF at +0.001. The crash and the subsequent rebound in 2020 favoured the rankings, while the rate-driven bear market of 2022 defeated them. Pooled into one regime, the two years cancel out, and the regime means (calm 0.005 against high volatility 0.004 for MASTER, for instance) hide the difference. Across universe-years, the mean RankIC is unrelated to the index volatility (correlation −0.06). What matters is the kind of market, not how much it moves.

**2. The portfolios follow the same split, and 2024 is the hardest year.** Against the equal-weight benchmark of the same days, MASTER, MATCC, FactorVAE and FinFormer gain 4–6% in 2020, while HIST and DiscoverPLF lose. Every model loses in 2024, from −4.7% to −10.7%. In the calm years only 30–44% of the runs beat the benchmark, which is consistent with a skill close to zero and the cost of concentrating on k stocks. In the volatile years the models split: MASTER (67% of runs), MATCC and FinFormer (53%) mostly beat it, and HIST and DiscoverPLF (20%) rarely do.

**3. The families the models rely on hardly change with the regime.** Between calm and volatile years, no family's net share moves by more than about three percentage points. The shifts are small but consistent within the Alpha158 group: all three models lose 1–3 points of momentum in the volatile years and gain 1.4–1.8 points of trend. HIST and DiscoverPLF lose 1.6–2.1 points of volume to momentum. FinFormer moves by less than one point.

**4. Explanations do not drift more when a volatile year is involved.** Between two consecutive test years of the same model, universe and seed, the families agree at 0.92 (calm to calm) against 0.93 (transitions into or out of a volatile year), and the features at 0.71 against 0.70. Per model, year-to-year feature agreement ranges from 0.95 (HIST) to 0.34 (FinFormer). For MASTER (0.72) and FactorVAE (0.56) it is far higher than their agreement across seeds in the same year (0.29 and 0.27, notebook 07). A model retrained one year later from the same seed stays closer to its earlier explanation than a model trained on the same data from another seed.

**Conclusion.** Regimes change what the models earn, not what they look at. Skill and returns differ sharply between 2020 and 2022, but the inputs the models rely on are the same in every year. This is consistent with models fitted on four-year windows that span several regimes, and with the diagnosis of notebook 07: the explanations are shaped by the weak, redundant signal of the inputs and by the training randomness, not by the market of the test year.